<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/split_wise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
from dataclasses import dataclass, field, replace
import heapq

@dataclass(order=True)
class Person:
    name: str = field(compare=False)
    debt: int = 0

    def __repr__(self):
        return f"{self.name}:{self.debt}"

@dataclass
class Txn:
    payer: Person
    recipient: Person
    amount: int

    def __repr__(self):
        return f"{self.payer.name} {self.amount}-> {self.recipient.name}"

class Pool:
    def __init__(self, pool=None):
        self.pool = pool or []

    def add_persons(self, *p):
        for x in p:
            self.pool.append(x)

    def spend(self, p, amount):
        p.debt -= amount
        for p in self.pool:
            p.debt += amount / len(self.pool)

    def __repr__(self):
        return ", ".join((repr(p) for p in self.pool))

    def greedy_settle(self):
        owe_heap = [p for p in self.pool if p.debt < 0]
        pay_heap = [replace(p, debt=-p.debt) for p in self.pool if p.debt > 0]
        heapq.heapify(owe_heap)
        heapq.heapify(pay_heap)
        txn = []
        while len(owe_heap):
            print(owe_heap,pay_heap)
            o, p = heapq.heappop(owe_heap), heapq.heappop(pay_heap)
            if o.debt > p.debt:
                txn.append(Txn(p, o, -o.debt))
                p.debt -= o.debt
                heapq.heappush(pay_heap, p)
            elif o.debt < p.debt:
                txn.append(Txn(p, o, -p.debt))
                o.debt -= p.debt
                heapq.heappush(owe_heap, o)
            else:
                txn.append(Txn(p, o, -o.debt))
        print(txn)

In [42]:
a, b, c = Person("A"), Person("B"), Person("C")
p = Pool()
p.add_persons(a,b,c)
p.spend(b,42)
print(p)
p.add_persons(Person("D"))
p.spend(a, 20)
print(p)
p.greedy_settle()

A:14.0, B:-28.0, C:14.0
A:-1.0, B:-23.0, C:19.0, D:5.0
[B:-23.0, A:-1.0] [C:-19.0, D:-5.0]
[B:-4.0, A:-1.0] [D:-5.0]
[A:-1.0] [D:-1.0]
[C 19.0-> B, D 4.0-> B, D 1.0-> A]


In [43]:
p = Pool([Person("A",-8),Person("B",5),Person("C",3),Person("D",-7),Person("E",7)])
p.greedy_settle()

[A:-8, D:-7] [E:-7, C:-3, B:-5]
[D:-7, A:-1] [B:-5, C:-3]
[D:-2, A:-1] [C:-3]
[A:-1] [C:-1]
[E 7-> A, B 5-> D, C 2-> D, C 1-> A]
